# Servidor de inferencia del peaje en Google Colab — entorno mamba

Este notebook levanta `inferencia/servidor_unificado.py` en un runtime GPU y publica la API para que Laravel la consuma. Laravel corre en el VPS (subdominio HTTPS); Flutter apunta a Laravel, no directamente a Colab. Ningún componente debe correr en tu PC durante la demo.

**Antes de empezar**
- En Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU** (T4 si está disponible).
- Ten a mano `inferencia/servidor_unificado.py` y `placas-web/python/placas/modelo/yolo-v9-t-384-license-plates-end2end.onnx` de este proyecto.
- Opcional: `m2_best.pt` entrenado para categorías y `llantas_best.pt` entrenado con keypoints de llantas. No uses un modelo de pose humana como M3.
- La URL de Cloudflare es temporal. Mantén este runtime activo durante las pruebas.

El endpoint está protegido con `X-API-Token`. No publiques el token ni compartas el notebook con el token guardado.

**Gestión de entornos:** este notebook crea el entorno **`peaje-inferencia` con micromamba** a partir de `inferencia/environment.yml` (versiones fijadas) y ejecuta el servidor **dentro de ese entorno** (`micromamba run`). Guarda además el diagnóstico del entorno (`diagnostico_colab_inferencia.txt`) y el lock con las versiones exactas (`environment.lock.yml`) en `MyDrive/peaje_modelos/evidencias_entornos/`.


In [ ]:
!nvidia-smi

In [ ]:
# Instala micromamba y crea el entorno conda/mamba "peaje-inferencia" (versiones fijadas).
# La primera vez tarda 5–10 min; queda en /content/mamba mientras dure la sesión.
import os, subprocess
from pathlib import Path

ENV_YML = r'''# Entorno conda/mamba del servidor de inferencia (M1 + M2 + M3 + pasajeros).
# Crear:   mamba env create -f environment.yml      (o: conda env create -f environment.yml)
# Activar: conda activate peaje-inferencia
# Versiones exactas del entorno ya creado: mamba env export -n peaje-inferencia > environment.lock.yml
name: peaje-inferencia
channels:
  - conda-forge
dependencies:
  - python=3.11
  - pip
  - tesseract
  - pip:
      - fastapi==0.115.6
      - uvicorn[standard]==0.34.0
      - python-multipart==0.0.20
      - numpy==1.26.4
      - torch==2.5.1
      - torchvision==0.20.1
      - opencv-python-headless==4.10.0.84
      - ultralytics==8.3.40
      - onnxruntime-gpu==1.20.0
      - easyocr==1.7.2
      - pytesseract==0.3.13
'''
Path('/content/environment.yml').write_text(ENV_YML)

if not Path('/usr/local/bin/micromamba').exists():
    subprocess.run('curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba',
                   shell=True, check=True)
MAMBA = ['micromamba', '-r', '/content/mamba']
if not Path('/content/mamba/envs/peaje-inferencia').exists():
    subprocess.run(MAMBA + ['create', '-y', '-q', '-n', 'peaje-inferencia', '-f', '/content/environment.yml'], check=True)
subprocess.run(MAMBA + ['env', 'list'], check=True)
ENV_RUN = MAMBA + ['run', '-n', 'peaje-inferencia']


In [ ]:
# Diagnóstico DENTRO del entorno mamba (evidencia de la bonificación de entornos)
DIAG = r'''
import sys, platform
print("Entorno       :", sys.prefix)
print("Python        :", sys.version.split()[0], platform.platform())
import torch
print("PyTorch       :", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
import ultralytics, cv2, onnxruntime as ort
print("Ultralytics   :", ultralytics.__version__)
print("OpenCV        :", cv2.__version__)
print("ONNX Runtime  :", ort.__version__, ort.get_available_providers())
'''
Path('/content/diag_env.py').write_text(DIAG)
salida = subprocess.run(ENV_RUN + ['python', '/content/diag_env.py'], capture_output=True, text=True)
print(salida.stdout, salida.stderr[-2000:])
Path('/content/diagnostico_colab_inferencia.txt').write_text(salida.stdout)
lock = subprocess.run(MAMBA + ['env', 'export', '-n', 'peaje-inferencia'], capture_output=True, text=True).stdout
Path('/content/environment.lock.yml').write_text(lock)
if 'CUDA: True' not in salida.stdout:
    raise RuntimeError('El entorno mamba no ve la GPU. Cambia el runtime a GPU (T4) y vuelve a ejecutar.')


## Cargar el servidor y los pesos (desde Google Drive)

Todos los archivos se buscan primero en **`MyDrive/peaje_modelos/`**. Si alguno falta, la celda pide subirlo y lo guarda en Drive para la próxima vez. Así, cuando Colab se reinicie, basta con volver a ejecutar las celdas.

| Archivo | Origen | Obligatorio |
|---|---|---|
| `servidor_unificado.py` | `inferencia/` del proyecto | Sí |
| `yolo-v9-t-384-license-plates-end2end.onnx` | `placas-web/python/placas/modelo/` (M1) | Sí |
| `m2_best.pt` | `entrenamiento_m2_segmentacion_vehiculos_roboflow.ipynb` (M2) | Sin él se usa COCO `yolo11n-seg.pt` |
| `llantas_best.pt` | `keypoints_llantas_yolo_detectron2_v3.ipynb` (M3) | Sin él los ejes se estiman por categoría |

Si cambias `servidor_unificado.py` en el proyecto, vuelve a subirlo a Drive (o bórralo de Drive para que la celda lo pida).


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil

drive.mount('/content/drive')
DIR_DRIVE = Path('/content/drive/MyDrive/peaje_modelos')
DIR_DRIVE.mkdir(parents=True, exist_ok=True)

def obtener(nombre, obligatorio, descripcion):
    # Copia el archivo desde Drive a /content; si falta, lo pide y lo guarda en Drive
    destino = Path('/content') / nombre
    en_drive = DIR_DRIVE / nombre
    if en_drive.is_file():
        shutil.copy(en_drive, destino)
        print(f'✓ {nombre} desde Drive ({en_drive.stat().st_size / 1e6:.1f} MB)')
        return destino
    if not obligatorio:
        r = input(f'No está {nombre} ({descripcion}) en Drive. ¿Subirlo ahora? [s/Enter=omitir]: ').strip().lower()
        if r not in ('s', 'si', 'sí'):
            print(f'– {nombre} omitido')
            return None
    print(f'Selecciona {nombre} ({descripcion})')
    subido = files.upload()
    if nombre not in subido:
        raise FileNotFoundError(f'Se esperaba un archivo llamado exactamente {nombre}.')
    destino.write_bytes(subido[nombre])
    shutil.copy(destino, en_drive)
    print(f'✓ {nombre} subido y guardado en Drive')
    return destino

servidor_path = obtener('servidor_unificado.py', True, 'carpeta inferencia del proyecto')
m1_path = obtener('yolo-v9-t-384-license-plates-end2end.onnx', True, 'modelo M1 de placas')
m2_path = obtener('m2_best.pt', False, 'modelo M2 de segmentación entrenado')
m3_path = obtener('llantas_best.pt', False, 'modelo M3 de keypoints de llantas')

print()
print('M2 propio:', m2_path is not None, '| M3 de llantas:', m3_path is not None)
if m2_path is None:
    print('AVISO: M2 usará yolo11n-seg (COCO), que no reconoce minibús. La rúbrica pide un modelo propio.')


# Evidencia de entornos en Drive
dir_ev = DIR_DRIVE / 'evidencias_entornos'
dir_ev.mkdir(exist_ok=True)
for f in ['diagnostico_colab_inferencia.txt', 'environment.lock.yml', 'environment.yml']:
    if Path('/content', f).exists():
        shutil.copy(Path('/content', f), dir_ev / f)
print('✓ Evidencia de entornos en', dir_ev)


## Configurar el token e iniciar la API

Introduce un secreto que también configurarás como `INFERENCE_API_TOKEN` en el `.env` de Laravel. La entrada es oculta y el notebook no imprime el valor. No guardes el valor en una celda de código.

In [ ]:
import getpass
import os
import subprocess
import sys
import time
import urllib.request
import json

token = getpass.getpass("INFERENCE_API_TOKEN (no se mostrará): ").strip()
if not token:
    raise ValueError("El token no puede estar vacío.")

os.environ["MODELO_M1_PATH"] = str(m1_path)
os.environ["MODELO_M2_PATH"] = str(m2_path) if m2_path else "yolo11n-seg.pt"
os.environ["MODELO_M3_PATH"] = str(m3_path) if m3_path else ""
os.environ["INFERENCE_API_TOKEN"] = token
os.environ["PORT"] = "8000"

log_servidor = open("/content/servidor-inferencia.log", "w")
servidor = subprocess.Popen(
    ENV_RUN + ['python', str(servidor_path)],
    cwd="/content",
    stdout=log_servidor,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

estado = None
limite = time.time() + 600
while time.time() < limite:
    if servidor.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/", timeout=3) as respuesta:
            estado = json.loads(respuesta.read().decode("utf-8"))
        break
    except Exception:
        time.sleep(3)

if estado is None:
    log_servidor.flush()
    print(Path("/content/servidor-inferencia.log").read_text(errors="replace")[-12000:])
    raise RuntimeError("La API no inició. Revisa el log anterior.")

print(json.dumps(estado, indent=2, ensure_ascii=False))
if not estado.get("cuda"):
    raise RuntimeError("El servidor inició sin CUDA disponible.")
if not estado.get("m1_cargado"):
    raise RuntimeError("M1 no se cargó. Revisa la ruta del modelo y el log del servidor.")
if not estado.get("m2_cargado"):
    raise RuntimeError("M2 no se cargó. Revisa la descarga o el peso configurado en el log.")
print("API lista en http://127.0.0.1:8000; M3 cargado:", estado.get("m3_cargado"))

## Crear una URL pública temporal

La URL se usa como `INFERENCE_URL` en Laravel, sin añadir `/predict`. El túnel expone el servicio a Internet; protege las solicitudes con el token y no compartas el token.

In [ ]:
import os
import re
import stat
import subprocess
import time
import urllib.request
from pathlib import Path

cloudflared_path = Path("/content/cloudflared")
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    cloudflared_path,
)
cloudflared_path.chmod(cloudflared_path.stat().st_mode | stat.S_IXUSR)

log_tunel = open("/content/cloudflared.log", "w")
tunel = subprocess.Popen(
    [str(cloudflared_path), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=log_tunel,
    stderr=subprocess.STDOUT,
)

url_publica = None
for _ in range(60):
    if tunel.poll() is not None:
        break
    time.sleep(2)
    log_tunel.flush()
    texto_log = Path("/content/cloudflared.log").read_text(errors="replace")
    coincidencia = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", texto_log)
    if coincidencia:
        url_publica = coincidencia.group(0)
        break

if not url_publica:
    log_tunel.flush()
    print(Path("/content/cloudflared.log").read_text(errors="replace")[-8000:])
    raise RuntimeError("No se obtuvo una URL de Cloudflare Tunnel.")

print("INFERENCE_URL=" + url_publica)
print("Configura INFERENCE_API_TOKEN en Laravel con el mismo secreto introducido anteriormente; no se muestra aquí.")

## Probar la inferencia de extremo a extremo

Sube una imagen de prueba. La respuesta debe incluir `placa`, `confianza`, `categoria`, `ejes` y `latencia_ms`.

In [ ]:
import requests
from google.colab import files

imagenes_prueba = files.upload()
if not imagenes_prueba:
    raise ValueError("Sube una imagen para probar la inferencia.")

nombre, contenido = next(iter(imagenes_prueba.items()))
respuesta = requests.post(
    url_publica + "/predict",
    headers={"X-API-Token": token},
    files={"file": (Path(nombre).name, contenido)},
    timeout=180,
)
print("HTTP", respuesta.status_code)
respuesta.raise_for_status()
print(json.dumps(respuesta.json(), indent=2, ensure_ascii=False))

## Conectar Laravel

En el VPS, en `/var/www/peaje/placas-web/.env`, configura la URL impresa y el mismo token que introdujiste de forma oculta:

```dotenv
INFERENCE_URL=https://tu-url.trycloudflare.com
INFERENCE_API_TOKEN=tu_secreto_privado
```

Luego, en el VPS, ejecuta `cd /var/www/peaje/placas-web && php artisan optimize:clear`. Comprueba en `https://tu-subdominio/estado` que la inferencia responde. Mantén Colab conectado mientras uses las funciones que llaman a inferencia. Al reiniciar el runtime, vuelve a ejecutar las celdas y actualiza la URL temporal en Laravel.

M1 (ONNX) y los modelos Ultralytics se cargan en GPU cuando sus proveedores CUDA están disponibles. Si `m3_cargado` es `false`, el servidor usa una estimación de ejes por categoría; no es una detección real de ruedas.